# 03 - Preprocessing dan Split Data
**Proyek:** Hotel Booking Cancellation Prediction | **Tahap:** M4 | **PRD:** v1.2

Tujuan: menyiapkan data yang siap dilatih **tanpa data leakage**.
Urutan penting: bersihkan data → **split dulu** → pelajari transformasi (encoding, scaling, kategori) **hanya dari data training**.

**Fitur (10 input aplikasi):** `lead_time`, `country`, `market_segment`, `total_of_special_requests`, `customer_type`, `hotel`, `nights`, `adr`, `previous_cancellations`, `is_repeated_guest`. **Target:** `is_canceled`.
Validasi ulang pemilihan fitur dilakukan di M5-M6 memakai 3 model dan cross-validation.

In [2]:
import os, glob, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

RANDOM_STATE = 42
os.makedirs('data/processed', exist_ok=True); os.makedirs('reports', exist_ok=True)

cands = (glob.glob('/kaggle/input/**/hotel_bookings.csv', recursive=True) 
         + glob.glob('./hotel_bookings.csv')
         + glob.glob('../hotel_bookings.csv')
         + glob.glob('./data/raw/hotel_bookings.csv') 
         + glob.glob('/mnt/user-data/uploads/hotel_bookings.csv'))
assert cands, 'hotel_bookings.csv tidak ditemukan'
raw = pd.read_csv(cands[0])
print('Data asli:', raw.shape)

Data asli: (119390, 32)


## 1. Pembersihan data

In [3]:
# 1) Hapus duplikat (PRD 7.5): semua kolom kecuali reservation_status_date
cols_dup = [c for c in raw.columns if c != 'reservation_status_date']
df = raw.drop_duplicates(subset=cols_dup).reset_index(drop=True)
print('Setelah hapus duplikat:', df.shape, '| dihapus:', len(raw) - len(df))

# 2) Fitur turunan + ambil hanya fitur terpilih + target. reservation_status*, parking, deposit_type dll. tidak dipakai.
df['nights'] = df.stays_in_weekend_nights + df.stays_in_week_nights
FEATURES = ['lead_time', 'country', 'market_segment', 'total_of_special_requests', 'customer_type',
         'hotel', 'nights', 'adr', 'previous_cancellations', 'is_repeated_guest']
CAT_FEATS = ['country', 'market_segment', 'customer_type', 'hotel']
NUM_FEATS = [f for f in FEATURES if f not in CAT_FEATS]
TARGET = 'is_canceled'
data = df[FEATURES + [TARGET]].copy()

print('\nMissing value pada fitur terpilih:'); print(data.isna().sum())
print('\nNilai tidak wajar (dicatat; ditangani oleh batas domain di pipeline, baris tidak dibuang):')
print('lead_time<0:', (data.lead_time < 0).sum(), '| special_requests<0:', (data.total_of_special_requests < 0).sum(),
      '| adr<0:', (data.adr < 0).sum(), '| adr>500:', (data.adr > 500).sum(), '| nights=0:', (data.nights == 0).sum(),
      '| nights>30:', (data.nights > 30).sum(), '| previous_cancellations>10:', (data.previous_cancellations > 10).sum())
print('market_segment langka:'); print(data.market_segment.value_counts().tail(3))

Setelah hapus duplikat: (87151, 32) | dihapus: 32239

Missing value pada fitur terpilih:
lead_time                      0
country                      451
market_segment                 0
total_of_special_requests      0
customer_type                  0
hotel                          0
nights                         0
adr                            0
previous_cancellations         0
is_repeated_guest              0
is_canceled                    0
dtype: int64

Nilai tidak wajar (dicatat; ditangani oleh batas domain di pipeline, baris tidak dibuang):
lead_time<0: 0 | special_requests<0: 0 | adr<0: 1 | adr>500: 3 | nights=0: 651 | nights>30: 23 | previous_cancellations>10: 39
market_segment langka:
market_segment
Complementary    702
Aviation         227
Undefined          2
Name: count, dtype: int64


**Keputusan:**
- **Missing value:** hanya `country` yang kosong (451 baris, ≈0,5%). Tidak dibuang, diberi kategori `Unknown` di dalam pipeline (jadi saat aplikasi menerima negara kosong tidak error). Fitur tambahan (`hotel`, `nights`, `adr`, `previous_cancellations`, `is_repeated_guest`) tidak memiliki missing value.
- **Nilai tidak wajar pada fitur baru:** `adr` negatif (sangat sedikit) dan sangat besar (maks 5.400), `nights` = 0 atau sangat panjang, `previous_cancellations` hingga 26. Baris **tidak dibuang**; nilai dibatasi dengan **batas domain tetap** di dalam pipeline: `adr` 0-500, `nights` 0-30, `previous_cancellations` 0-10. Batas ini aturan domain (bukan dihitung dari data), sehingga aman dari leakage, dan sama persis dengan rentang input di aplikasi.
- **`adr = 0`:** dipertahankan (umumnya reservasi komplimen), bukan error.
- **Outlier `lead_time`** (maksimum 737 hari): tidak dibuang karena pemesanan jauh-jauh hari adalah data yang valid. Model pohon tidak sensitif terhadap outlier; untuk Logistic Regression dipakai `log1p` untuk meredam kemiringan (juga untuk `nights` dan `previous_cancellations`).

## 2. Split train/test (dilakukan sebelum transformasi apa pun)

In [4]:
X = data[FEATURES]; y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

split_info = pd.DataFrame({
    'jumlah baris': [len(X_train), len(X_test)],
    '% batal': [round(y_train.mean()*100, 2), round(y_test.mean()*100, 2)]},
    index=['train', 'test'])
display(split_info)
assert len(set(X_train.index) & set(X_test.index)) == 0, 'Ada baris yang sama di train dan test!'
print('Tidak ada baris yang sama di train dan test: OK')

pd.concat([X_train, y_train], axis=1).to_csv('data/processed/train.csv', index=True)
pd.concat([X_test, y_test], axis=1).to_csv('data/processed/test.csv', index=True)

,jumlah baris,% batal
train,69720,27.29
test,17431,27.28


Tidak ada baris yang sama di train dan test: OK


**Interpretasi:** split 80:20 *stratified* menjaga proporsi kelas batal tetap sama pada train dan test (≈27%), sehingga evaluasi tidak bias. `random_state=42` membuat hasil dapat direproduksi. Data test **disimpan terpisah dan tidak disentuh** sampai evaluasi akhir (M6).

## 3. Pipeline preprocessing

In [5]:
# Batas domain TETAP (winsorizing, bukan dipelajari dari data): meredam nilai ekstrem tanpa membuang baris
CLIP = {'adr': (0, 500), 'nights': (0, 30), 'previous_cancellations': (0, 10)}
LOG_FEATS = ['lead_time', 'nights', 'previous_cancellations']   # miring ke kanan -> log1p (khusus Logistic Regression)
NUM_ALL = ['lead_time', 'total_of_special_requests', 'nights', 'guests', 'adr', 'previous_cancellations', 'is_repeated_guest']
log_tf = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

def clip_tf(lo, hi):
    return FunctionTransformer(np.clip, kw_args={'a_min': lo, 'a_max': hi}, feature_names_out='one-to-one')

def num_pipe(n, linear):
    steps = []
    if n in CLIP: steps.append(('clip', clip_tf(*CLIP[n])))
    if linear:
        if n in LOG_FEATS: steps.append(('log', log_tf))
        if n != 'is_repeated_guest': steps.append(('sc', StandardScaler()))
    return Pipeline(steps) if steps else 'passthrough'

def build_prep(feats, linear=False):
    """ColumnTransformer untuk set fitur tertentu. linear=True -> clip + log1p + scaling (Logistic Regression)."""
    nums = [f for f in feats if f in NUM_ALL]
    cats = [f for f in feats if f not in NUM_ALL]
    tfs = []
    if 'country' in cats:   # 10 negara teratas di TRAIN punya kolom sendiri, sisanya (termasuk Unknown) -> 'lainnya'
        tfs.append(('country', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
                    ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist', max_categories=11, sparse_output=False))]), ['country']))
        cats = [c for c in cats if c != 'country']
    if cats:
        tfs.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cats))
    for n in nums:
        tfs.append((n, num_pipe(n, linear), [n]))
    return ColumnTransformer(tfs)

# Dua pipeline: pohon (Random Forest, XGBoost) tanpa scaling; linear (Logistic Regression) dengan clip + log1p + scaling.
# log1p dan clip tidak punya parameter yang dipelajari dari data, jadi aman dari leakage.
prep_tree = build_prep(FEATURES, linear=False)
prep_linear = build_prep(FEATURES, linear=True)

# fit HANYA pada data training
prep_tree.fit(X_train); prep_linear.fit(X_train)
Xt_tree = prep_tree.transform(X_train); Xt_lin = prep_linear.transform(X_train)
print('Bentuk X_train hasil transformasi -> tree:', Xt_tree.shape, '| linear:', Xt_lin.shape)

names = prep_tree.get_feature_names_out()
print(len(names), 'fitur setelah encoding:'); print(list(names))

Bentuk X_train hasil transformasi -> tree: (69720, 31) | linear: (69720, 31)
31 fitur setelah encoding:
['country__country_BEL', 'country__country_BRA', 'country__country_DEU', 'country__country_ESP', 'country__country_FRA', 'country__country_GBR', 'country__country_IRL', 'country__country_ITA', 'country__country_NLD', 'country__country_PRT', 'country__country_infrequent_sklearn', 'cat__market_segment_Aviation', 'cat__market_segment_Complementary', 'cat__market_segment_Corporate', 'cat__market_segment_Direct', 'cat__market_segment_Groups', 'cat__market_segment_Offline TA/TO', 'cat__market_segment_Online TA', 'cat__market_segment_Undefined', 'cat__customer_type_Contract', 'cat__customer_type_Group', 'cat__customer_type_Transient', 'cat__customer_type_Transient-Party', 'cat__hotel_City Hotel', 'cat__hotel_Resort Hotel', 'lead_time__lead_time', 'total_of_special_requests__total_of_special_requests', 'nights__nights', 'adr__adr', 'previous_cancellations__previous_cancellations', 'is_repeat

**Alasan tiap langkah (siapkan untuk tanya jawab):**
- **Imputer `Unknown`:** nilai kosong pada `country` dijadikan kategori sendiri, bukan dihapus.
- **One-Hot Encoding:** `country`, `market_segment`, `customer_type`, `hotel` adalah kategori tanpa urutan, sehingga tidak boleh diberi angka 1,2,3 (model akan menganggap ada urutan).
- **`max_categories=11` pada `country`:** hanya 10 negara teratas di data training yang diberi kolom sendiri, sisanya (termasuk `Unknown`) digabung ke satu kolom "lainnya". Mencegah ±170 kolom jarang dan cocok dengan dropdown aplikasi (top-10 + Lainnya).
- **Clip batas domain (`adr`, `nights`, `previous_cancellations`):** meredam nilai ekstrem tanpa membuang data; batas tetap, tidak dipelajari dari data.
- **`log1p` + scaling hanya untuk Logistic Regression:** regresi linier sensitif pada skala fitur, sedangkan Random Forest dan XGBoost berbasis pohon sehingga tidak butuh scaling. `is_repeated_guest` sudah biner 0/1 sehingga tidak di-scale.
- **Fit hanya di train:** rata-rata, simpangan baku, dan daftar kategori dipelajari dari data training saja, lalu diterapkan ke test. Inilah yang mencegah leakage.

## 4. Pengecekan hasil transformasi

In [6]:
# Kategori yang dipelajari dari TRAIN
oh = prep_tree.named_transformers_['country'].named_steps['oh']
kept = [c for c in oh.categories_[0] if c not in (oh.infrequent_categories_[0] if oh.infrequent_categories_[0] is not None else [])]
print('Negara dengan kolom sendiri (%d):' % len(kept), kept)
print('Negara digabung ke "lainnya":', len(oh.infrequent_categories_[0]))
cat_oh = prep_tree.named_transformers_['cat']
for col, cats in zip(['market_segment', 'customer_type', 'hotel'], cat_oh.categories_):
    print(f'{col:15s}:', list(cats))

# Statistik hasil scaling pada TRAIN (kolom ter-scaling harus mean~0, std~1)
lin_out = pd.DataFrame(Xt_lin, columns=prep_linear.get_feature_names_out())
scaled = [c for c in lin_out.columns if c.split('__')[0] in ['lead_time', 'total_of_special_requests', 'nights', 'adr', 'previous_cancellations']]
print('\nMean & std kolom ter-scaling (train):'); print(lin_out[scaled].agg(['mean', 'std']).round(3).T)

# Cek batas domain bekerja: nilai ekstrem harus terpotong di pipeline pohon
tr_out = pd.DataFrame(Xt_tree, columns=names)
print('\nadr setelah clip  -> min/max:', tr_out['adr__adr'].min(), tr_out['adr__adr'].max())
print('nights setelah clip -> min/max:', tr_out['nights__nights'].min(), tr_out['nights__nights'].max())
print('prev_cancellations setelah clip -> min/max:', tr_out['previous_cancellations__previous_cancellations'].min(), tr_out['previous_cancellations__previous_cancellations'].max())

# Uji ketahanan: input dengan negara baru/kosong tidak boleh error (ini yang terjadi di aplikasi)
sample = pd.DataFrame([{'lead_time': 120, 'total_of_special_requests': 1, 'country': 'XXX', 'market_segment': 'Online TA',
                        'customer_type': 'Transient', 'hotel': 'City Hotel', 'nights': 3, 'adr': 95.0,
                        'previous_cancellations': 0, 'is_repeated_guest': 0},
                       {'lead_time': 5, 'total_of_special_requests': 0, 'country': np.nan, 'market_segment': 'Direct',
                        'customer_type': 'Contract', 'hotel': 'Resort Hotel', 'nights': 400, 'adr': 9999.0,
                        'previous_cancellations': 50, 'is_repeated_guest': 1}])
print('\nUji input baru/ekstrem -> bentuk:', prep_tree.transform(sample[FEATURES]).shape, '(tidak error: OK)')

# Cek tidak ada NaN setelah transformasi
assert not np.isnan(prep_tree.transform(X_test)).any() and not np.isnan(prep_linear.transform(X_test)).any()
print('Transformasi test tanpa NaN: OK')

Negara dengan kolom sendiri (10): ['BEL', 'BRA', 'DEU', 'ESP', 'FRA', 'GBR', 'IRL', 'ITA', 'NLD', 'PRT']
Negara digabung ke "lainnya": 160
market_segment : ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups', 'Offline TA/TO', 'Online TA', 'Undefined']
customer_type  : ['Contract', 'Group', 'Transient', 'Transient-Party']
hotel          : ['City Hotel', 'Resort Hotel']

Mean & std kolom ter-scaling (train):
                                                    mean  std
lead_time__lead_time                                 0.0  1.0
total_of_special_requests__total_of_special_req...   0.0  1.0
nights__nights                                      -0.0  1.0
adr__adr                                             0.0  1.0
previous_cancellations__previous_cancellations      -0.0  1.0

adr setelah clip  -> min/max: 0.0 500.0
nights setelah clip -> min/max: 0.0 30.0
prev_cancellations setelah clip -> min/max: 0.0 10.0

Uji input baru/ekstrem -> bentuk: (2, 31) (tidak error: OK)
Transformasi

**Interpretasi:** setelah fit di train, kolom ter-scaling punya rata-rata ≈ 0 dan simpangan baku ≈ 1. Nilai ekstrem `adr`, `nights`, dan `previous_cancellations` terpotong pada batas domain (cek min/max di atas). Input dengan negara yang belum pernah terlihat (`XXX`) atau kosong tetap bisa ditransformasi tanpa error, karena digabung ke kategori "lainnya". Ini penting karena pengguna aplikasi bisa memilih negara apa pun. Transformasi pada data test tidak menghasilkan NaN.

## 5. Menyimpan konfigurasi

In [7]:
config = {'features': FEATURES, 'num_feats': NUM_FEATS, 'cat_feats': CAT_FEATS, 'target': TARGET,
          'random_state': RANDOM_STATE, 'test_size': 0.2,
          'clip_bounds': {k: list(v) for k, v in CLIP.items()},
          'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
          'market_segment_values': sorted(X_train.market_segment.unique().tolist()),
          'customer_type_values': sorted(X_train.customer_type.unique().tolist()),
          'hotel_values': sorted(X_train.hotel.unique().tolist()),
          'top_countries': X_train.country.value_counts().head(10).index.tolist()}
with open('reports/preprocessing_config.json', 'w') as f: json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

{
  "features": [
    "lead_time",
    "country",
    "market_segment",
    "total_of_special_requests",
    "customer_type",
    "hotel",
    "nights",
    "adr",
    "previous_cancellations",
    "is_repeated_guest"
  ],
  "num_feats": [
    "lead_time",
    "total_of_special_requests",
    "nights",
    "adr",
    "previous_cancellations",
    "is_repeated_guest"
  ],
  "cat_feats": [
    "country",
    "market_segment",
    "customer_type",
    "hotel"
  ],
  "target": "is_canceled",
  "random_state": 42,
  "test_size": 0.2,
  "clip_bounds": {
    "adr": [
      0,
      500
    ],
    "nights": [
      0,
      30
    ],
    "previous_cancellations": [
      0,
      10
    ]
  },
  "n_train": 69720,
  "n_test": 17431,
  "market_segment_values": [
    "Aviation",
    "Complementary",
    "Corporate",
    "Direct",
    "Groups",
    "Offline TA/TO",
    "Online TA",
    "Undefined"
  ],
  "customer_type_values": [
    "Contract",
    "Group",
    "Transient",
    "Transient-Party"


## 6. Kesimpulan M4 (v1.2)
1. Duplikat dihapus (32.239 baris) → 87.151 baris; hanya 10 fitur + target yang dipakai (`nights` dibuat dari `stays_in_weekend_nights + stays_in_week_nights`).
2. Split 80:20 stratified (`random_state=42`) dilakukan **sebelum** transformasi; test disimpan terpisah.
3. Dua pipeline dari satu fungsi `build_prep`: `prep_tree` (Random Forest, XGBoost) dan `prep_linear` (Logistic Regression, dengan clip + log1p + scaling), semuanya di-fit hanya pada train.
4. `country` kosong → `Unknown`; hanya top-10 negara diberi kolom sendiri; nilai ekstrem `adr`/`nights`/`previous_cancellations` dibatasi batas domain tetap; input baru/ekstrem tidak membuat error.
5. Class imbalance (≈27% batal) tidak ditangani dengan resampling. Di M5 pembobotan kelas hanya diuji sebagai pembanding; keputusan akhir di M6 adalah **tanpa `class_weight`**, dan trade-off Precision/Recall diatur lewat threshold.

**Langkah berikutnya: M5 Modeling** (`04_model_training.ipynb`).